# 09 · Reddit Sentiment Labeling

LLM-based feature extraction for Reddit FX posts.  
Labels each post with 7 structured features using `gpt-oss-120b` via Groq.

**Designed for Google Colab.** Colab-specific cells (Drive mount, pip install) are marked — skip them when running locally.

In [ ]:
import json, time, shutil
from pathlib import Path
from datetime import datetime

import pandas as pd
from tqdm.auto import tqdm
import google.generativeai as genai
from google.colab import userdata

print("imports ok")

imports ok


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
# Colab only — skip if running locally
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 2.5 MB/s eta 0:00:00


In [ ]:
from groq import Groq, RateLimitError
print("groq ok")

groq ok


In [ ]:
from google.colab import drive
import os

# Mount drive if not already mounted
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

# Install unrar if necessary (usually pre-installed in Colab)
# !apt-get install unrar

# Create a target directory for extraction
extract_path = '/content/reddit_data'
os.makedirs(extract_path, exist_ok=True)

# Extract the .rar file
!unrar x "/content/drive/MyDrive/reddit.rar" "{extract_path}/"

Mounted at /content/drive

UNRAR 6.11 beta 1 freeware      Copyright (c) 1993-2022 Alexander Roshal


Extracting from /content/drive/MyDrive/reddit.rar

Extracting  /content/reddit_data/economics_raw_20260405_210126.jsonl       2%  5%  8% 11% 13% 14%  OK 
Extracting  /content/reddit_data/forex_raw_20260405_192900.jsonl          17% 20% 23% 25% 28% 31% 34% 36%  OK 
Extracting  /content/reddit_data/investing_raw_20260405_210126.jsonl      39% 42% 45% 48% 50% 53% 56% 59% 62%  OK 
Extracting  /content/reddit_data/stocks_raw_20260405_210126.jsonl         65% 67% 70% 73% 76% 79% 81% 84% 87% 90% 93% 95% 98%100%  OK 
All OK


In [ ]:
# Update DATA_DIR to point to the extracted files
DATA_DIR = Path("/content/reddit_data")
print(f"Updated DATA_DIR: {DATA_DIR}")
print(f"Files found: {[p.name for p in DATA_DIR.glob('*.jsonl')]}")

Updated DATA_DIR: /content/reddit_data
Files found: ['investing_raw_20260405_210126.jsonl', 'stocks_raw_20260405_210126.jsonl', 'forex_raw_20260405_192900.jsonl', 'economics_raw_20260405_210126.jsonl']


In [ ]:

# ── API Keys ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# You can mix Groq keys (gsk_...), Google Gemini keys, and OpenRouter keys (sk-or-...)
API_KEYS = [
    "gsk_REPLACE_ME_1",
    # "gsk_REPLACE_ME_2",
]

# Automatically add Google key from Colab secrets if available
# try:
#     secret_google = userdata.get('GOOGLE_API_KEY')
#     if secret_google and secret_google not in API_KEYS:
#         API_KEYS.append(secret_google)
#         print("Added GOOGLE_API_KEY from secrets to pool")
# except:
#     pass

# ── Paths ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
DATA_DIR        = Path("/content/reddit_data")
OUTPUT_DIR      = Path("/content/drive/MyDrive/FX-AlphaLab/processed/sentiment_labels")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_FILE = OUTPUT_DIR / "reddit_labels_checkpoint.jsonl"

# ── Model ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Provider-specific model assignments
GROQ_MODEL = "openai/gpt-oss-120b"
GEMINI_MODEL = "gemma-4-26b-a4b-it"
OPENROUTER_MODEL = "qwen/qwen3-14b"

# ── Subreddits ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SUBREDDITS = ["Forex", "investing", "stocks"]

# ── Run parameters ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SAMPLE_SIZE      = None
CHECKPOINT_EVERY = 40
MAX_BODY_CHARS   = 1200
MAX_RETRIES      = 5

print("config ok")


config ok


## Data Exploration

In [ ]:
# Load one file raw to inspect actual field values before deciding anything
def load_jsonl(path: Path) -> pd.DataFrame:
    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return pd.DataFrame(records)

df_forex = load_jsonl(DATA_DIR / "forex_raw_20260405_192900.jsonl")
df_forex["subreddit"] = "Forex"
print(f"Shape: {df_forex.shape}")
print(f"\nColumns: {list(df_forex.columns)}")

Shape: (112809, 123)

Columns: ['all_awardings', 'allow_live_comments', 'archived', 'author', 'author_created_utc', 'author_flair_background_color', 'author_flair_css_class', 'author_flair_template_id', 'author_flair_text', 'author_flair_text_color', 'can_gild', 'category', 'content_categories', 'contest_mode', 'created_utc', 'discussion_type', 'distinguished', 'domain', 'edited', 'gilded', 'gildings', 'hidden', 'hide_score', 'id', 'is_created_from_ads_ui', 'is_crosspostable', 'is_meta', 'is_original_content', 'is_reddit_media_domain', 'is_robot_indexable', 'is_self', 'is_video', 'link_flair_background_color', 'link_flair_css_class', 'link_flair_richtext', 'link_flair_template_id', 'link_flair_text', 'link_flair_text_color', 'link_flair_type', 'locked', 'media', 'media_embed', 'media_only', 'name', 'no_follow', 'num_comments', 'num_crossposts', 'over_18', 'parent_whitelist_status', 'permalink', 'pinned', 'pwls', 'quarantine', 'removed_by_category', 'retrieved_utc', 'score', 'secure_med

In [ ]:
frames = []
for sr in SUBREDDITS:
    matches = sorted(DATA_DIR.glob(f"{sr.lower()}_raw_*.jsonl"))
    path = matches[-1]
    print(f"Loading r/{sr} from {path.name} ...")
    df = load_jsonl(path)
    df["subreddit"] = sr
    frames.append(df)
    print(f"  {len(df):,} rows, {df.shape[1]} cols")

df_all = pd.concat(frames, ignore_index=True)
print(f"\nCombined: {df_all.shape}")

Loading r/Forex from forex_raw_20260405_192900.jsonl ...
  112,809 rows, 123 cols
Loading r/investing from investing_raw_20260405_210126.jsonl ...
  203,178 rows, 124 cols
Loading r/stocks from stocks_raw_20260405_210126.jsonl ...
  257,669 rows, 123 cols

Combined: (573656, 126)


In [ ]:
print("=== removed_by_category value counts per subreddit ===\n")
for sr, grp in df_all.groupby("subreddit"):
    counts = grp["removed_by_category"].value_counts(dropna=False)
    total = len(grp)
    print(f"r/{sr}  ({total:,} posts)")
    for val, n in counts.items():
        print(f"  {str(val):<30} {n:>7,}  ({n/total:.1%})")
    print()

=== removed_by_category value counts per subreddit ===

r/Forex  (112,809 posts)
  None                            57,083  (50.6%)
  moderator                       25,587  (22.7%)
  reddit                          21,845  (19.4%)
  deleted                          8,277  (7.3%)
  content_takedown                    14  (0.0%)
  copyright_takedown                   3  (0.0%)

r/investing  (203,178 posts)
  moderator                      111,019  (54.6%)
  None                            55,834  (27.5%)
  deleted                         23,252  (11.4%)
  reddit                          13,047  (6.4%)
  content_takedown                    23  (0.0%)
  author                               3  (0.0%)

r/stocks  (257,669 posts)
  moderator                      142,063  (55.1%)
  None                            75,710  (29.4%)
  deleted                         30,566  (11.9%)
  reddit                           9,298  (3.6%)
  content_takedown                    29  (0.0%)
  author            

**r/Forex has the best survival rate — 50.6% not removed.** r/investing and r/stocks are heavily moderated (54–55% moderator-removed). Only `removed_by_category == "None"` posts are viable for labeling — that gives ~57k Forex, ~56k investing, ~76k stocks (~189k total). The `"None"` value is a string, not Python None — need to filter on the string.

In [ ]:
print("=== author field — special values ===\n")
for sr, grp in df_all.groupby("subreddit"):
    special = grp["author"].value_counts()
    special = special[special.index.isin(["[deleted]", "[removed]", "AutoModerator", None])]
    total = len(grp)
    print(f"r/{sr}  ({total:,} posts)")
    for val, n in special.items():
        print(f"  {str(val):<30} {n:>7,}  ({n/total:.1%})")
    null_n = grp["author"].isna().sum()
    print(f"  {'NaN/null':<30} {null_n:>7,}  ({null_n/total:.1%})")
    print()

=== author field — special values ===

r/Forex  (112,809 posts)
  [deleted]                        9,998  (8.9%)
  NaN/null                             0  (0.0%)

r/investing  (203,178 posts)
  [deleted]                       28,239  (13.9%)
  AutoModerator                    2,154  (1.1%)
  NaN/null                             0  (0.0%)

r/stocks  (257,669 posts)
  [deleted]                       37,641  (14.6%)
  AutoModerator                    2,070  (0.8%)
  NaN/null                             0  (0.0%)



**`[deleted]` authors are significant (8–15%) and must be dropped** — their posts have no attributable stance. AutoModerator appears only in investing/stocks (~1%) and is pure noise. No null authors exist, so no null-check needed. Filter: drop `[deleted]` and `AutoModerator`.

In [ ]:
print("=== title field ===\n")
for sr, grp in df_all.groupby("subreddit"):
    null_n = grp["title"].isna().sum()
    empty_n = (grp["title"].astype(str).str.strip() == "").sum()
    lens = grp["title"].astype(str).str.len()
    print(f"r/{sr}")
    print(f"  null:          {null_n:,}")
    print(f"  empty string:  {empty_n:,}")
    print(f"  len  min/p5/median/p95/max:  {lens.min()} / {lens.quantile(.05):.0f} / {lens.median():.0f} / {lens.quantile(.95):.0f} / {lens.max()}")
    print(f"  len < 10:      {(lens < 10).sum():,}")
    print()

=== title field ===

r/Forex
  null:          0
  empty string:  0
  len  min/p5/median/p95/max:  1 / 8 / 33 / 123 / 300
  len < 10:      6,877

r/investing
  null:          0
  empty string:  0
  len  min/p5/median/p95/max:  1 / 25 / 46 / 116 / 303
  len < 10:      615

r/stocks
  null:          0
  empty string:  0
  len  min/p5/median/p95/max:  1 / 14 / 39 / 107 / 300
  len < 10:      923



In [ ]:
# What do the very short titles actually look like?
short = df_all[df_all["title"].astype(str).str.len() < 10][["subreddit", "title"]].head(20)
print(short.to_string(index=False))

subreddit     title
    Forex      US30
    Forex Giving Up
    Forex   bye bye
    Forex     Oanda
    Forex    Advice
    Forex      FXCM
    Forex    Update
    Forex   EUR/USD
    Forex     Taxes
    Forex   EUR/AUD
    Forex   Eur/usd
    Forex beginners
    Forex     Books
    Forex $ 500Easy
    Forex  Question
    Forex     STINU
    Forex   Returns
    Forex      HODL
    Forex  Charges?
    Forex  A EURUSD


**No null titles across any subreddit.** Short titles (< 10 chars) are mostly one-word posts — "Advice", "Question", "EUR/USD" — that carry no standalone sentiment signal without a body. There are 6,877 such posts in r/Forex (6%), 615 in investing, 923 in stocks. These should be dropped unless they have a meaningful body.

In [ ]:
print("=== selftext (body) field ===\n")
for sr, grp in df_all.groupby("subreddit"):
    total = len(grp)
    null_n = grp["selftext"].isna().sum()
    body = grp["selftext"].astype(str).str.strip()
    deleted_n = (body == "[deleted]").sum()
    removed_n = (body == "[removed]").sum()
    empty_n   = (body == "").sum()
    has_real  = total - null_n - deleted_n - removed_n - empty_n
    real_lens = grp["selftext"].astype(str).str.len()
    real_lens = real_lens[~body.isin({"[deleted]", "[removed]", ""})]
    print(f"r/{sr}  ({total:,} posts)")
    print(f"  null:            {null_n:>7,}  ({null_n/total:.1%})")
    print(f"  '[deleted]':     {deleted_n:>7,}  ({deleted_n/total:.1%})")
    print(f"  '[removed]':     {removed_n:>7,}  ({removed_n/total:.1%})")
    print(f"  empty string:    {empty_n:>7,}  ({empty_n/total:.1%})")
    print(f"  has real body:   {has_real:>7,}  ({has_real/total:.1%})")
    if len(real_lens):
        print(f"  body len median/p95: {real_lens.median():.0f} / {real_lens.quantile(.95):.0f}")
    print()

=== selftext (body) field ===

r/Forex  (112,809 posts)
  null:                  0  (0.0%)
  '[deleted]':       5,151  (4.6%)
  '[removed]':      40,813  (36.2%)
  empty string:     24,042  (21.3%)
  has real body:    42,803  (37.9%)
  body len median/p95: 245 / 1335

r/investing  (203,178 posts)
  null:                  0  (0.0%)
  '[deleted]':       2,946  (1.4%)
  '[removed]':     147,867  (72.8%)
  empty string:          6  (0.0%)
  has real body:    52,359  (25.8%)
  body len median/p95: 595 / 2465

r/stocks  (257,669 posts)
  null:                  0  (0.0%)
  '[deleted]':       7,120  (2.8%)
  '[removed]':     179,608  (69.7%)
  empty string:         58  (0.0%)
  has real body:    70,883  (27.5%)
  body len median/p95: 594 / 3802



**Body is mostly absent or junk.** r/Forex has 37.9% real body text (median 245 chars), which aligns with the EDA findings. r/investing and r/stocks have 25–28% real bodies but they're longer (median ~595 chars) when present. The dominant junk values are `"[removed]"` (36–73%) and `"[deleted]"` (1–5%) — both must be treated as empty. Empty string is rare except in r/Forex (21%, likely link posts). **The body enriches but is never required** — the title is the primary text for labeling.

In [ ]:
# Short-title posts that DO have a real body — are they worth keeping?
short_mask = df_all["title"].astype(str).str.len() < 10
junk_body  = {"[deleted]", "[removed]", ""}
has_body   = ~df_all["selftext"].astype(str).str.strip().isin(junk_body)

short_with_body = df_all[short_mask & has_body][["subreddit", "title", "selftext"]].head(10)
for _, row in short_with_body.iterrows():
    print(f"[r/{row['subreddit']}] title: {row['title']!r}")
    print(f"  body: {str(row['selftext'])[:200]}")
    print()

[r/Forex] title: 'Giving Up'
  body: I just wanted to share my experience during the past 5 years since I started reading and exploring about Forex. just a little about me. I come from Engineering background and I started my career as a 

[r/Forex] title: 'Oanda'
  body: I am currently demo trading on Oanda. I can always enter trades, long or short. But when I try to close or modify the trade with a TP or SL it always gives me the "Off Quotes" error. Is this a common 

[r/Forex] title: 'EUR/AUD'
  body: Any thoughts on EUR/AUD guys. Entered a mayor s block

[r/Forex] title: 'Eur/usd'
  body: Eur/usd is about to shoot up bullish watch could potentially see it at 123 area again

[r/Forex] title: 'beginners'
  body: How long did you take to study enough to know how to trade and get decent profit? (while on work and going to school)


edit : newbie here

[r/Forex] title: 'Question'
  body: Anyone know  any good forex discord?

[r/Forex] title: 'Returns'
  body: Hypothetically speaking, how

**Short-title posts with a body are often signal-bearing.** "EUR/AUD — Any thoughts, entered a major s block" or "Eur/usd is about to shoot up bullish" — the title alone is < 10 chars but the body carries a clear directional stance. **Drop the MIN_TITLE_LEN filter.** Instead, drop posts where BOTH title < 10 chars AND body is empty/junk — those have no usable text at all.

In [ ]:
print("=== link_flair_text coverage and top values ===\n")
for sr, grp in df_all.groupby("subreddit"):
    total = len(grp)
    has_flair = grp["link_flair_text"].notna() & (grp["link_flair_text"].astype(str) != "None")
    print(f"r/{sr}  — flair coverage: {has_flair.sum():,} / {total:,}  ({has_flair.mean():.1%})")
    top = grp.loc[has_flair, "link_flair_text"].value_counts().head(8)
    for val, n in top.items():
        print(f"  {str(val):<35} {n:>6,}")
    print()

=== link_flair_text coverage and top values ===

r/Forex  — flair coverage: 111,272 / 112,809  (98.6%)
  Questions                           41,650
  Charts and Setups                   27,359
  Prop Firms                           9,152
  Fundamental Analysis                 9,022
  OTHER/META                           8,900
  P/L Porn                             7,137
  Brokers                              6,266
  MEMES                                1,777

r/investing  — flair coverage: 898 / 203,178  (0.4%)
  Daily Discussion                       743
  News                                    50
  Potentially Misleading or Incorrect     30
  Asset Management                        20
  DD / Fundamentals                       11
  Misleading or Incorrect Title            9
  Macro                                    8
  r/investing                              6

r/stocks  — flair coverage: 120,543 / 257,669  (46.8%)
  Advice Request                      32,515
  Advice              

**Flair is only reliable in r/Forex (98.6%).** r/investing has near-zero flair (0.4%) — don't include it in the prompt for those posts. r/stocks has 46.8% flair coverage. For r/Forex, flair is a strong signal for the LLM: "Charts and Setups" → likely TECHNICAL, "Fundamental Analysis" → FUNDAMENTAL, "P/L Porn" → POSITION_DISCLOSURE, "MEMES" → NOISE. Include flair in the prompt only when present.

In [ ]:
col = df_all["removed_by_category"]
print("dtype:", col.dtype)
print("null count:", col.isna().sum())
print("unique values:", col.unique()[:10])
print("type of first non-null:", type(col.dropna().iloc[0]))
print("repr of first non-null:", repr(col.dropna().iloc[0]))

dtype: object
null count: 188627
unique values: ['deleted' None 'reddit' 'moderator' 'copyright_takedown'
 'content_takedown' 'author' 'automod_filtered']
type of first non-null: <class 'str'>
repr of first non-null: 'deleted'


**`removed_by_category` is actual NaN (not the string "None") when the post was not removed.** pandas printed it as `None` in the earlier value_counts output. The correct filter is `.isna()`, not `== "None"`.

In [ ]:
junk_body = {"[deleted]", "[removed]", ""}

keep = (
    # 1. Not removed by anyone (NaN = not removed)
    df_all["removed_by_category"].isna()
    # 2. Real author
    & (~df_all["author"].isin({"[deleted]", "AutoModerator"}))
    # 3. Has usable text: title >= 10 chars OR a real body
    & (
        (df_all["title"].astype(str).str.len() >= 10)
        | (~df_all["selftext"].astype(str).str.strip().isin(junk_body))
    )
)

print("Posts surviving all filters:\n")
for sr, grp in df_all.groupby("subreddit"):
    sr_keep = keep[grp.index].sum()
    print(f"r/{sr:<12} {sr_keep:>7,}  /  {len(grp):>7,}  ({sr_keep/len(grp):.1%})")

print(f"\nTotal: {keep.sum():,} / {len(df_all):,}  ({keep.mean():.1%})")

Posts surviving all filters:

r/Forex         54,755  /  112,809  (48.5%)
r/investing     49,434  /  203,178  (24.3%)
r/stocks        67,752  /  257,669  (26.3%)

Total: 171,941 / 573,656  (30.0%)


**171,941 posts survive (30% of raw).** r/Forex retains 48.5%, investing and stocks ~24–26% due to heavy moderation. This is the labeling corpus. Now build the clean dataframe and verify a sample before writing any LLM code.

In [ ]:
df_clean = df_all[keep][["id", "subreddit", "title", "selftext", "link_flair_text", "score", "created_utc"]].copy()

# Normalise body: collapse all junk to empty string
junk_body = {"[deleted]", "[removed]", ""}
df_clean["body"] = (
    df_clean["selftext"]
    .astype(str)
    .str.strip()
    .apply(lambda x: "" if x in junk_body else x)
)
df_clean = df_clean.drop(columns=["selftext"])

# Normalise flair: NaN → empty string
df_clean["flair"] = df_clean["link_flair_text"].fillna("").astype(str).str.strip()
df_clean = df_clean.drop(columns=["link_flair_text"])

df_clean = df_clean.reset_index(drop=True)

print(f"Shape: {df_clean.shape}")
print(f"\nColumns: {list(df_clean.columns)}")
print(f"\nBody coverage (non-empty):")
print(df_clean.groupby("subreddit")["body"].apply(lambda x: (x != "").sum()).rename("has_body"))
print(f"\nFlair coverage (non-empty):")
print(df_clean.groupby("subreddit")["flair"].apply(lambda x: (x != "").sum()).rename("has_flair"))

Shape: (171941, 7)

Columns: ['id', 'subreddit', 'title', 'score', 'created_utc', 'body', 'flair']

Body coverage (non-empty):
subreddit
Forex        42172
investing    49427
stocks       67716
Name: has_body, dtype: int64

Flair coverage (non-empty):
subreddit
Forex        54714
investing       61
stocks       39105
Name: has_flair, dtype: int64


In [ ]:
# Spot-check: one sample from each subreddit
for sr, grp in df_clean.groupby("subreddit"):
    row = grp.sample(1, random_state=42).iloc[0]
    print(f"[r/{sr}]  flair={row['flair']!r}")
    print(f"  title: {row['title']}")
    if row["body"]:
        print(f"  body:  {row['body'][:250]}")
    print()

[r/Forex]  flair='OTHER/META'
  title: For you
  body:  Guys don’t ever give up the market gonna frustrate you but with consistency you will be profitable

[r/investing]  flair='Potentially Misleading or Incorrect'
  title: Money Market Risk in a stock market crash.  Maybe long term treasuries are better.
  body:  Money Markets are very low risk but they have been tested in the past and have been “rescued” by the federal government in 2008 when they “broke the buck”.

It’s been some time since then and I had forgotten about a little bit of a bank run situation

[r/stocks]  flair='Company News'
  title: These are the stocks on my watchlist (4/18)
  body:  Hi! I am an ex-prop trader that trades equities.

  
This is a daily watchlist for trading. I might trade all of the stocks on here, or none of them, on any given day. I might trade stocks that don't appear on here! I hold no positions in any stocks



## Additional Filtering — Pre-labeling Quality Check

Five candidate filters to remove posts that carry no signal and would waste API tokens.  
Explore each distribution first — thresholds are set in the application cell below.

In [ ]:
# ── Filter Exploration 1/5: Score distribution ────────────────────────────────
# Question: does a score threshold eliminate low-quality noise without cutting real signal?
# Hypothesis: score <= 0 posts are downvoted or invisible — community rejected them.

print("=== Score distribution per subreddit ===\n")
for sr, grp in df_clean.groupby("subreddit"):
    total = len(grp)
    s = grp["score"]
    print(f"r/{sr}  ({total:,} posts)")
    print(
        f"  min / p5 / p25 / median / p75 / p95 / max: "
        f"{s.min()} / {s.quantile(.05):.0f} / {s.quantile(.25):.0f} / "
        f"{s.median():.0f} / {s.quantile(.75):.0f} / {s.quantile(.95):.0f} / {s.max()}"
    )
    for threshold in [0, 1, 2, 3, 5]:
        n = (s <= threshold).sum()
        print(f"  score <= {threshold:<1}: {n:>6,}  ({n/total:.1%})")
    print()

# Spot-check: what do score <= 0 posts look like?
print("=== Samples: score <= 0 ===")
low_score = df_clean[df_clean["score"] <= 0][["subreddit", "score", "title", "body"]].head(8)
for _, row in low_score.iterrows():
    print(f"  [r/{row['subreddit']}] score={row['score']}  {row['title']!r}")
    if row["body"]:
        print(f"    body: {row['body'][:120]}")

=== Score distribution per subreddit ===

r/Forex  (54,755 posts)
  min / p5 / p25 / median / p75 / p95 / max: 0 / 0 / 1 / 2 / 7 / 48 / 1031
  score <= 0:  6,288  (11.5%)
  score <= 1: 21,809  (39.8%)
  score <= 2: 29,513  (53.9%)
  score <= 3: 34,412  (62.8%)
  score <= 5: 39,133  (71.5%)

r/investing  (49,434 posts)
  min / p5 / p25 / median / p75 / p95 / max: 0 / 0 / 0 / 1 / 11 / 243 / 26836
  score <= 0: 16,992  (34.4%)
  score <= 1: 24,984  (50.5%)
  score <= 2: 27,992  (56.6%)
  score <= 3: 30,143  (61.0%)
  score <= 5: 33,059  (66.9%)

r/stocks  (67,752 posts)
  min / p5 / p25 / median / p75 / p95 / max: 0 / 0 / 1 / 6 / 34 / 560 / 88310
  score <= 0: 12,778  (18.9%)
  score <= 1: 19,692  (29.1%)
  score <= 2: 23,714  (35.0%)
  score <= 3: 27,058  (39.9%)
  score <= 5: 32,150  (47.5%)

=== Samples: score <= 0 ===
  [r/Forex] score=0  'Signal Providers'
    body: I’ve been speaking to 2 different signal

https://marketpulse365.com/

[Hanford & Company](https://hanfordandcompany.co

In [ ]:
# ── Filter Exploration 2/5: URL-only bodies ───────────────────────────────────
# Question: how many posts have a body that is nothing but a bare URL?
# These posts carry zero textual signal beyond the title.

url_only_mask = df_clean["body"].str.match(r"^https?://\S+$", na=False)

print(f"=== URL-only bodies ===\n")
print(f"Total:  {url_only_mask.sum():,}  ({url_only_mask.mean():.2%} of df_clean)\n")
for sr, grp in df_clean.groupby("subreddit"):
    n = url_only_mask[grp.index].sum()
    print(f"r/{sr:<12} {n:>5,}  ({n/len(grp):.2%})")

print("\n--- Samples ---")
samples = df_clean[url_only_mask][["subreddit", "score", "title", "body"]].head(8)
for _, row in samples.iterrows():
    print(f"  [r/{row['subreddit']}] score={row['score']}  title={row['title']!r}")
    print(f"    body: {row['body']}")

=== URL-only bodies ===

Total:  195  (0.11% of df_clean)

r/Forex          188  (0.34%)
r/investing        2  (0.00%)
r/stocks           5  (0.01%)

--- Samples ---
  [r/Forex] score=178  title="My first scammer - I didn't think he'd keep going"
    body: https://imgur.com/gallery/lFeUvzI
  [r/Forex] score=2  title='Understanding pips'
    body: https://www.reddit.com/r/Daytrading/comments/lc6g03/understanding_pips/?utm_source=share&utm_medium=ios_app&utm_name=iossmf
  [r/Forex] score=0  title='Do you prefer to trade currencies or indices? POLL'
    body: https://linkto.run/p/J1GPIJR3
  [r/Forex] score=3  title="Hello I'm new to this stuff... Does anyone know if this a ligit website? Thank you🙂"
    body: https://linkscapital.ltd/index.php?a=home
  [r/Forex] score=0  title='I expect a continued drop for GBPAUD'
    body: https://www.tradingview.com/x/WdNSfyuF/
  [r/Forex] score=1  title='GBPAUD short idea'
    body: https://preview.redd.it/0brnelimtpd81.png?width=1834&format=png&auto=

In [ ]:
# ── Filter Exploration 3/5: Duplicate titles ──────────────────────────────────
# Question: how many posts share an exact (normalised) title?
# Recurring automated titles (daily discussions, weekly threads) are pure noise.

title_norm = df_clean["title"].str.lower().str.strip()
dup_counts = title_norm.value_counts()
dups = dup_counts[dup_counts > 1]

total_dup_posts = dups.sum()
print(f"=== Duplicate titles ===\n")
print(f"Unique titles that appear more than once: {len(dups):,}")
print(f"Posts sharing a duplicate title:          {total_dup_posts:,}  ({total_dup_posts/len(df_clean):.1%})\n")

print("Top 25 most duplicated titles:")
for title, count in dups.head(25).items():
    # show which subreddits
    subs = df_clean.loc[title_norm == title, "subreddit"].value_counts().to_dict()
    print(f"  {count:>4}x  {title!r}  {subs}")

# Distribution: how many titles appear exactly 2x, 3x, …?
print("\nDuplicate frequency distribution:")
freq_dist = dups.value_counts().sort_index()
for n_copies, n_titles in freq_dist.items():
    posts = n_copies * n_titles
    print(f"  {n_copies:>3}x: {n_titles:>5,} titles  ({posts:,} posts)")

=== Duplicate titles ===

Unique titles that appear more than once: 4,239
Posts sharing a duplicate title:          11,542  (6.7%)

Top 25 most duplicated titles:
   116x  'xauusd'  {'Forex': 116}
    87x  'help'  {'Forex': 87}
    71x  'question'  {'Forex': 69, 'stocks': 2}
    61x  'prop firms'  {'Forex': 61}
    59x  'gold'  {'Forex': 59}
    55x  'forex'  {'Forex': 55}
    49x  'eurusd'  {'Forex': 49}
    36x  'gbpusd'  {'Forex': 36}
    35x  'ftmo'  {'Forex': 35}
    32x  'funded account'  {'Forex': 32}
    32x  'eur/usd'  {'Forex': 32}
    32x  'new to forex'  {'Forex': 32}
    31x  'backtesting'  {'Forex': 31}
    29x  'gbpjpy'  {'Forex': 29}
    28x  'need help'  {'Forex': 27, 'stocks': 1}
    28x  'strategy'  {'Forex': 28}
    25x  'us30'  {'Forex': 25}
    25x  'risk management'  {'Forex': 23, 'stocks': 2}
    24x  'wash sale question'  {'stocks': 24}
    24x  'prop firm'  {'Forex': 24}
    23x  'advice'  {'Forex': 21, 'investing': 1, 'stocks': 1}
    23x  'forex trading'  {'

In [ ]:
# ── Filter Exploration 4/5: Combined text length ──────────────────────────────
# Question: where is the "too short to label" cliff?
# A post with 15 combined chars (e.g. title="EUR/USD", body="") carries no stance.

combined_len = (df_clean["title"] + " " + df_clean["body"]).str.len()

print("=== Combined text length (title + body chars) ===\n")
for sr, grp in df_clean.groupby("subreddit"):
    total = len(grp)
    cl = combined_len[grp.index]
    print(f"r/{sr}  ({total:,} posts)")
    print(
        f"  min / p1 / p5 / p10 / p25 / median: "
        f"{cl.min()} / {cl.quantile(.01):.0f} / {cl.quantile(.05):.0f} / "
        f"{cl.quantile(.10):.0f} / {cl.quantile(.25):.0f} / {cl.median():.0f}"
    )
    for threshold in [15, 20, 25, 30, 40, 50]:
        n = (cl < threshold).sum()
        print(f"  combined < {threshold:<2} chars: {n:>5,}  ({n/total:.1%})")
    print()

print("--- Samples: combined < 20 chars ---")
very_short = df_clean[combined_len < 20][["subreddit", "title", "body"]].head(12)
for _, row in very_short.iterrows():
    print(f"  [r/{row['subreddit']}]  title={row['title']!r}  body={row['body']!r}")

=== Combined text length (title + body chars) ===

r/Forex  (54,755 posts)
  min / p1 / p5 / p10 / p25 / median: 3 / 14 / 25 / 38 / 83 / 205
  combined < 15 chars:   602  (1.1%)
  combined < 20 chars: 1,510  (2.8%)
  combined < 25 chars: 2,627  (4.8%)
  combined < 30 chars: 3,653  (6.7%)
  combined < 40 chars: 5,816  (10.6%)
  combined < 50 chars: 7,873  (14.4%)

r/investing  (49,434 posts)
  min / p1 / p5 / p10 / p25 / median: 23 / 293 / 326 / 356 / 443 / 630
  combined < 15 chars:     0  (0.0%)
  combined < 20 chars:     0  (0.0%)
  combined < 25 chars:     1  (0.0%)
  combined < 30 chars:     1  (0.0%)
  combined < 40 chars:     4  (0.0%)
  combined < 50 chars:     6  (0.0%)

r/stocks  (67,752 posts)
  min / p1 / p5 / p10 / p25 / median: 19 / 235 / 270 / 304 / 404 / 630
  combined < 15 chars:     0  (0.0%)
  combined < 20 chars:     1  (0.0%)
  combined < 25 chars:     3  (0.0%)
  combined < 30 chars:     4  (0.0%)
  combined < 40 chars:    11  (0.0%)
  combined < 50 chars:    21  (

In [ ]:
# ── Filter Exploration 5/5: High-frequency authors ───────────────────────────
# Question: are there bot-like or spam accounts posting at abnormal rates?
# These inflate one voice into thousands of labeled samples — bad for training diversity.

# author is not in df_clean — join from df_all via post id
author_lookup = df_all.set_index("id")["author"]
author_series = df_clean["id"].map(author_lookup)

author_counts = author_series.value_counts()
total_posts   = len(df_clean)

print("=== Top 40 most prolific authors in df_clean ===\n")
print(f"{'Rank':<5} {'Author':<35} {'Posts':>6}  {'% corpus':>9}  {'Subreddits'}")
print("-" * 80)
for rank, (author, count) in enumerate(author_counts.head(40).items(), 1):
    subs = df_clean[author_series == author]["subreddit"].value_counts().to_dict()
    pct  = count / total_posts
    flag = " ⚑" if count >= 100 else ""
    print(f"{rank:<5} {str(author):<35} {count:>6,}  {pct:>8.2%}  {subs}{flag}")

print(f"\nTotal unique authors: {author_counts.nunique():,}")
print(f"Authors with >= 50 posts:  {(author_counts >= 50).sum():,}")
print(f"Authors with >= 100 posts: {(author_counts >= 100).sum():,}")
print(f"Authors with >= 200 posts: {(author_counts >= 200).sum():,}")

# Posts captured by authors with >= 100 posts
heavy = author_counts[author_counts >= 100]
heavy_posts = heavy.sum()
print(f"\nPosts from authors >= 100 posts: {heavy_posts:,}  ({heavy_posts/total_posts:.2%})")

=== Top 40 most prolific authors in df_clean ===

Rank  Author                               Posts   % corpus  Subreddits
--------------------------------------------------------------------------------
1     Puginator                              930     0.54%  {'stocks': 930} ⚑
2     bigbear0083                            506     0.29%  {'stocks': 506} ⚑
3     WinningWatchlist                       328     0.19%  {'stocks': 328} ⚑
4     apooroldinvestor                       289     0.17%  {'stocks': 285, 'investing': 4} ⚑
5     WickedSensitiveCrew                    270     0.16%  {'stocks': 269, 'investing': 1} ⚑
6     _hiddenscout                           268     0.16%  {'stocks': 267, 'investing': 1} ⚑
7     ivuser1                                263     0.15%  {'Forex': 263} ⚑
8     Progress_8                             237     0.14%  {'stocks': 228, 'investing': 9} ⚑
9     psychotrader00                         208     0.12%  {'stocks': 208} ⚑
10    callsonreddit             

### Filter Decisions

| # | Filter | Finding | Decision |
|---|--------|---------|----------|
| 1 | **Score ≤ 0** | Removes 11–34% depending on sub. Score=0 posts have real content — early posts get 0 votes before anyone sees them. Score is a popularity signal, not a quality signal. | **Skip** |
| 2 | **URL-only body** | 195 posts (0.11%). Body is a bare URL — zero textual signal. | **Apply** |
| 3 | **Duplicate titles** | 6.7% of posts. Duplicates are lazy re-titling ("XAUUSD", "Help"), NOT automated threads. Each has a distinct body. Deduplication by title would silently drop real signal. | **Skip** |
| 4 | **Short combined text** | Only meaningful in r/Forex (p5 = 25 chars vs 270+ in the others). Posts < 20 combined chars are meme/image posts with no body. Samples confirm they're unlabelable. | **Apply** — drop if `len(title + body) < 20` |
| 5 | **High-frequency authors** | Top author has 930 posts (0.54%). These are active users and daily watchlist posters — not bots. However, allowing >200 posts per author would let a single voice account for ~0.5% of the labeled dataset. Cap at 200 to prevent extreme over-representation without cutting genuine contributors. | **Apply** — cap at 200 posts per author |

Net effect: removes URL-only posts, degenerate short posts, and the tail of author over-representation.

In [ ]:
# ── Apply pre-labeling filters ────────────────────────────────────────────────
# Filter 1: URL-only bodies  (body is bare URL, no textual signal)
# Filter 2: Short combined text  (title + body < 20 chars)
# Filter 3: Author post cap  (> 200 posts per author → sample down to 200)

# Reconstruct author series (needed for cap)
author_series = df_clean["id"].map(df_all.set_index("id")["author"])
df_clean = df_clean.copy()
df_clean["_author"] = author_series

n_before = len(df_clean)
print(f"Before filters: {n_before:,}\n")

# Filter 1 — URL-only bodies
url_only  = df_clean["body"].str.match(r"^https?://\S+$", na=False)
df_clean  = df_clean[~url_only]
print(f"After URL-only filter:       {len(df_clean):,}  (removed {url_only.sum():,})")

# Filter 2 — Short combined text
combined_len  = (df_clean["title"] + " " + df_clean["body"]).str.len()
too_short     = combined_len < 20
df_clean      = df_clean[~too_short]
print(f"After short-text filter:     {len(df_clean):,}  (removed {too_short.sum():,})")

# Filter 3 — Author cap at 200 posts
author_counts  = df_clean["_author"].value_counts()
capped_authors = author_counts[author_counts > 200].index
keep_idx = []
for author in capped_authors:
    mask = df_clean["_author"] == author
    sampled = df_clean[mask].sample(200, random_state=42).index
    keep_idx.extend(sampled.tolist())

# Keep all non-capped rows + sampled rows for capped authors
not_capped_mask = ~df_clean["_author"].isin(capped_authors)
df_clean = pd.concat([
    df_clean[not_capped_mask],
    df_clean.loc[keep_idx],
], ignore_index=True)
n_capped_removed = (author_counts[author_counts > 200] - 200).sum()
print(f"After author cap (>200):     {len(df_clean):,}  (removed {n_capped_removed:,})")

df_clean = df_clean.drop(columns=["_author"])
df_clean = df_clean.reset_index(drop=True)

print(f"\nTotal removed: {n_before - len(df_clean):,}  ({(n_before - len(df_clean))/n_before:.2%})")
print(f"Final df_clean: {len(df_clean):,} posts\n")
print("Per-subreddit breakdown:")
print(df_clean["subreddit"].value_counts())

Before filters: 171,941

After URL-only filter:       171,746  (removed 195)
After short-text filter:     170,235  (removed 1,511)
After author cap (>200):     168,737  (removed 1,498)

Total removed: 3,204  (1.86%)
Final df_clean: 168,737 posts

Per-subreddit breakdown:
subreddit
stocks       66313
Forex        52995
investing    49429
Name: count, dtype: int64


## Balanced Sampling

Weighted round-robin interleaving: **50% r/Forex / 25% r/investing / 25% r/stocks**.

FX-specific content dominates at 50% because the downstream model targets FX sentiment.  
Round-robin by fractional position ensures the target distribution holds at any early stopping point.

In [ ]:
# ── Weighted round-robin interleaving ─────────────────────────────────────────
# Target: 50% Forex / 25% investing / 25% stocks
#
# Method: assign each post a global "slot" = rank_within_subreddit / weight.
# At any stopping point k, subreddit s contributes k * weight_s posts,
# guaranteeing the target distribution regardless of where the run is interrupted.

WEIGHTS = {"Forex": 0.50, "investing": 0.25, "stocks": 0.25}

frames = []
for sr, weight in WEIGHTS.items():
    subset = (
        df_clean[df_clean["subreddit"] == sr]
        .sample(frac=1, random_state=42)
        .reset_index(drop=True)
    )
    # Global slot: post i of subreddit s appears at position i / weight_s.
    # Tie-break with a tiny per-subreddit offset so identical slots sort deterministically.
    tie_break = list(WEIGHTS.keys()).index(sr) * 1e-9
    subset["_pos"] = subset.index / weight + tie_break
    frames.append(subset)

df_labeling = (
    pd.concat(frames, ignore_index=True)
    .sort_values("_pos")
    .drop(columns=["_pos"])
    .reset_index(drop=True)
)

print(f"df_labeling shape: {df_labeling.shape}")
print(f"\nTarget weights: {WEIGHTS}")
print("\nActual composition at different cut-points:")
for n in [1_000, 5_000, 10_000, 25_000, 50_000, len(df_labeling)]:
    if n > len(df_labeling):
        break
    counts = df_labeling.head(n)["subreddit"].value_counts()
    breakdown = "  ".join(f"{sr}: {counts.get(sr, 0)/n:.1%}" for sr in WEIGHTS)
    print(f"  first {n:>7,}: {breakdown}")

df_labeling shape: (168737, 7)

Target weights: {'Forex': 0.5, 'investing': 0.25, 'stocks': 0.25}

Actual composition at different cut-points:
  first   1,000: Forex: 50.0%  investing: 25.0%  stocks: 25.0%
  first   5,000: Forex: 50.0%  investing: 25.0%  stocks: 25.0%
  first  10,000: Forex: 50.0%  investing: 25.0%  stocks: 25.0%
  first  25,000: Forex: 50.0%  investing: 25.0%  stocks: 25.0%
  first  50,000: Forex: 50.0%  investing: 25.0%  stocks: 25.0%
  first 168,737: Forex: 31.4%  investing: 29.3%  stocks: 39.3%


class KeyPool:
    """
    Manages a mixed pool of Groq and Google Gemini API keys.
    Detects provider from key prefix and handles backoff globally.
    """
    MAX_BACKOFF = 300.0

    def __init__(self, api_keys: list[str]):
        self.keys = list(set(api_keys))
        self.clients = {}
        self.available_after = {k: 0.0 for k in self.keys}
        self.fail_count = {k: 0 for k in self.keys}
        self.provider = {}

        for k in self.keys:
            if k.startswith("gsk_"):
                self.provider[k] = "groq"
                self.clients[k] = Groq(api_key=k)
            else:
                self.provider[k] = "google"
                # Clients for Gemini are usually instantiated per request via genai.GenerativeModel
                # so we just store the key here.
                self.clients[k] = k

    def _earliest(self) -> tuple[str, float]:
        key = min(self.keys, key=lambda k: self.available_after[k])
        wait = max(0.0, self.available_after[key] - time.time())
        return key, wait

    def acquire(self) -> tuple[str, str, any]:
        key, wait = self._earliest()
        if wait > 0:
            print(f"  [pool] all keys in backoff — waiting {wait:.1f}s")
            time.sleep(wait)
        return key, self.provider[key], self.clients[key]

    def on_success(self, key: str):
        self.fail_count[key] = 0

    def on_rate_limit(self, key: str, retry_after: float | None = None):
        self.fail_count[key] += 1
        backoff = retry_after if retry_after else min(
            10.0 * (2 ** self.fail_count[key]),
            self.MAX_BACKOFF,
        )
        self.available_after[key] = time.time() + backoff
        print(f"  [pool] key ...{key[-6:]} ({self.provider[key]}) limited — backoff {backoff:.0f}s")

print("Mixed KeyPool defined")

In [ ]:
import openai
import threading

class KeyPool:
    """
    Thread-safe pool of Groq, Google Gemini, and OpenRouter API keys.
    Detects provider from key prefix and handles per-key exponential backoff.
    Safe to call from multiple threads simultaneously.
    """
    MAX_BACKOFF = 300.0

    def __init__(self, api_keys: list[str]):
        self._lock = threading.Lock()
        self.keys = list(set(api_keys))
        self.clients = {}
        self.available_after = {k: 0.0 for k in self.keys}
        self.fail_count = {k: 0 for k in self.keys}
        self.provider = {}

        for k in self.keys:
            if k.startswith("gsk_"):
                self.provider[k] = "groq"
                self.clients[k] = Groq(api_key=k)
            elif k.startswith("sk-or-"):
                self.provider[k] = "openrouter"
                self.clients[k] = openai.OpenAI(
                    base_url="https://openrouter.ai/api/v1",
                    api_key=k,
                )
            else:
                self.provider[k] = "google"
                self.clients[k] = k

    def _earliest_unsafe(self) -> tuple[str, float]:
        # Must be called with self._lock held
        key = min(self.keys, key=lambda k: self.available_after[k])
        wait = max(0.0, self.available_after[key] - time.time())
        return key, wait

    def acquire(self) -> tuple[str, str, any]:
        while True:
            with self._lock:
                key, wait = self._earliest_unsafe()
            if wait <= 0:
                return key, self.provider[key], self.clients[key]
            # Sleep outside the lock so other threads can still acquire available keys
            time.sleep(min(wait, 0.5))

    def on_success(self, key: str):
        with self._lock:
            self.fail_count[key] = 0

    def on_rate_limit(self, key: str, retry_after: float | None = None):
        with self._lock:
            self.fail_count[key] += 1
            backoff = retry_after if retry_after else min(
                10.0 * (2 ** self.fail_count[key]),
                self.MAX_BACKOFF,
            )
            self.available_after[key] = time.time() + backoff
        print(f"  [pool] key ...{key[-6:]} ({self.provider[key]}) limited — backoff {backoff:.0f}s")

print("Thread-safe KeyPool defined")

Thread-safe KeyPool defined


In [ ]:
import time
import threading
from unittest.mock import MagicMock, patch

# Verify backoff logic without making any API calls

with patch("groq.Groq", return_value=MagicMock()):
    pool = KeyPool(["key_aaa", "key_bbb"])

# Simulate key_aaa hitting rate limit
pool.on_rate_limit("key_aaa", retry_after=60)
print(f"key_aaa available_after set: {pool.available_after['key_aaa'] - time.time():.1f}s from now")
print(f"key_aaa fail_count: {pool.fail_count['key_aaa']}")

# _earliest should now return key_bbb (available immediately)
key, wait = pool._earliest_unsafe()
print(f"\n_earliest_unsafe() -> key={key!r}, wait={wait:.2f}s")
assert key == "key_bbb", "should pick the available key"

# Simulate key_bbb also hitting rate limit
pool.on_rate_limit("key_bbb", retry_after=30)
key, wait = pool._earliest_unsafe()
print(f"_earliest_unsafe() after both limited -> key={key!r}, wait={wait:.1f}s  (key_bbb has shorter backoff)")
assert key == "key_bbb"

# on_success resets fail_count
pool.on_success("key_aaa")
print(f"\nAfter on_success: key_aaa fail_count={pool.fail_count['key_aaa']}")

# Exponential backoff without retry_after: 10 * 2^n
print("\nTesting Exponential Backoff:")
pool2_fail = KeyPool(["k"])
pool2_fail.available_after = {"k": 0.0}
pool2_fail.fail_count = {"k": 0}

for i in range(4):
    pool2_fail.available_after["k"] = 0.0  # reset so we can call multiple times
    pool2_fail.on_rate_limit("k")
    expected = min(10.0 * (2 ** pool2_fail.fail_count["k"]), KeyPool.MAX_BACKOFF)
    print(f"  fail #{pool2_fail.fail_count['k']} -> backoff={expected:.0f}s")

print("\nAll KeyPool assertions passed")

  [pool] key ...ey_aaa (google) limited — backoff 60s
key_aaa available_after set: 60.0s from now
key_aaa fail_count: 1

_earliest_unsafe() -> key='key_bbb', wait=0.00s
  [pool] key ...ey_bbb (google) limited — backoff 30s
_earliest_unsafe() after both limited -> key='key_bbb', wait=30.0s  (key_bbb has shorter backoff)

After on_success: key_aaa fail_count=0

Testing Exponential Backoff:
  [pool] key ...k (google) limited — backoff 20s
  fail #1 -> backoff=20s
  [pool] key ...k (google) limited — backoff 40s
  fail #2 -> backoff=40s
  [pool] key ...k (google) limited — backoff 80s
  fail #3 -> backoff=80s
  [pool] key ...k (google) limited — backoff 160s
  fail #4 -> backoff=160s

All KeyPool assertions passed


**KeyPool logic verified.** Rate-limited keys are parked correctly, `_earliest()` always picks the soonest-available key, exponential backoff scales 10→20→40→80s without API calls, and `on_success` resets the fail counter.

## Prompt & label extraction

In [ ]:
SYSTEM_PROMPT = """You are a financial market sentiment labeler. You extract structured labels from Reddit posts to train a lightweight FX sentiment classification model.

## Decision sequence

1. Determine `content_type`.
2. Assess `sarcasm_irony_score`. If score=2, the literal sentiment is inverted: force sentiment_strength=0 and stance_clarity=NONE.
3. Determine `stance_clarity`: if any question is asked ("thoughts?", "should I?", "any tips?", "help me improve?"), use QUESTION regardless of content_type.
4. Fill all remaining fields.

## Output format

Return a single JSON object with exactly these 8 fields. No other text.

{
  "reasoning":           <max 15 words: content_type + key observation about stance or sentiment>,
  "content_type":        <"TECHNICAL" | "FUNDAMENTAL" | "NEWS_REACTION" | "POSITION_DISCLOSURE" | "NOISE">,
  "sarcasm_irony_score": <integer: 0, 1, or 2>,
  "sentiment_strength":  <integer: -2, -1, 0, 1, or 2>,
  "target_pair":         <string or null>,
  "risk_sentiment":      <"RISK_ON" | "RISK_OFF" | "NEUTRAL">,
  "target_clarity":      <integer: 0, 1, or 2>,
  "stance_clarity":      <"CLEAR" | "CONDITIONAL" | "QUESTION" | "NONE">
}

All integers must be integers, not floats.

## Field definitions

### reasoning
Max 15 words. Identify the content_type and your key observation about stance or sentiment. Written before you commit to any label — guides subsequent fields.

### content_type
What the post is fundamentally about. Assign based on dominant content:
- TECHNICAL: Price action, trends, support/resistance, indicators (RSI, MACD, Bollinger), candlestick analysis — on any asset. "Following the downtrend" or "wicking below the 200 MA" is TECHNICAL regardless of flair label.
- FUNDAMENTAL: Macro forces only — central bank policy, inflation, GDP, interest rates, geopolitical drivers. If the post discusses price levels, trends, or chart patterns, use TECHNICAL instead. Single-stock corporate actions (earnings, debt restructuring, M&A, company milestones) → NOISE.
- NEWS_REACTION: Direct reaction to a specific named recent event ("Fed just hiked 75bps today", "tariffs announced this morning", "NFP just dropped"). General discussion of Fed policy = FUNDAMENTAL; "Fed hiked yesterday, I'm now short USD" = NEWS_REACTION.
- POSITION_DISCLOSURE: Author reveals an already-executed live or recently completed trade — entry, exit, P&L, or open position size. An author considering or planning a trade is NOT POSITION_DISCLOSURE. When both chart analysis and an open position are present, POSITION_DISCLOSURE takes priority. Empty-body posts where flair (P/L Porn, Charts and Setups) and title reference a trade result or open position qualify.
- NOISE: No market signal. Memes, broker/platform comparisons, prop firm questions, account setup, tax/legal questions, personal posts with no market reference. NOT NOISE: posts discussing a specific trade, chart, market instrument, or macro theme — use the appropriate content_type with stance_clarity=QUESTION.

Important: content_type captures the topic, not the author's posture. A question about EUR/USD fundamentals is FUNDAMENTAL, not NOISE. Use stance_clarity=QUESTION for posts that ask rather than state.

### sarcasm_irony_score
- 0: No sarcasm or irony.
- 1: Ambiguous — could be sarcastic ("great, another rate hike, very bullish").
- 2: Clear sarcasm — literal sentiment reading is inverted. Force sentiment_strength=0 and stance_clarity=NONE. Includes "X they said", mock praise after a loss, bitter irony ("Love this market 🙄").

### sentiment_strength
Direction and magnitude of market sentiment toward the target pair/currency:
- +2: Strongly bullish, high conviction.
- +1: Mildly bullish, directional lean without strong conviction.
-  0: No directional view. Use when stance_clarity is QUESTION or NONE, or sarcasm_irony_score=2.
- -1: Mildly bearish.
- -2: Strongly bearish, high conviction.

### target_pair
The currency pair or currency the sentiment applies to. Infer from context when implicit.
- Standard notation: "going long EUR/USD" → "EURUSD", "short cable" → "GBPUSD"
- Colloquial: "cable" → "GBPUSD", "fiber" → "EURUSD", "loonie" → "CAD", "kiwi" → "NZD", "swissy" → "CHF", "Aussie" → "AUDUSD", "greenback"/"dollar" → "USD", "yen" → "JPY", "euro" → "EUR"
- **Non-FX assets (stocks, indices, crypto, commodities): always null** — they still contribute risk_sentiment.
- null when no currency reference exists, or when a currency is mentioned only for context with no directional sentiment (e.g., observing market hours, platform issues).
- **Hard rule: target_clarity must be 0 whenever target_pair is null.**

### risk_sentiment
Overall macro/broad market risk tone — scope is macro only. Single-stock earnings or CEO news without systemic implications = NEUTRAL.
- RISK_ON: Appetite for risk assets. Falling inflation, rate cut expectations, growth optimism, markets rallying, high-yield/EM currencies favoured (AUD, NZD, MXN, BRL). Long USD/EM pairs = RISK_OFF, not RISK_ON.
- RISK_OFF: Flight to safety. Recession fears, geopolitical shock, systemic fraud/market manipulation, market crash, safe-haven demand (JPY, CHF, USD). Short EM currencies or long safe havens signals RISK_OFF.
- NEUTRAL: No macro risk tone. Isolated technical analysis, routine position updates, company-specific news, no macro context.

### target_clarity
How explicitly the currency target is stated:
- 0: No currency mentioned and none inferable. Must be 0 when target_pair is null.
- 1: Implied via colloquial name or indirect reference ("cable", "the yen", "Aussie", "the dollar").
- 2: Explicitly stated in standard notation ("EUR/USD", "EURUSD", "JPY", "USD").

### stance_clarity
How the author presents their view:
- CLEAR: Direct, unambiguous directional market claim ("I'm short EUR/USD", "strong buy signal here"). Requires an explicit directional statement — sharing P&L results, an account update, or a chart without a stated market view is not CLEAR (use NONE).
- CONDITIONAL: Hedged or if/then stance ("if EUR breaks 1.10 I'll go long", "could go either way after NFP").
- QUESTION: Author solicits opinions or asks for feedback — "thoughts?", "any tips?", "should I hold?", "help me improve?". Use whenever any question is asked, regardless of content_type.
- NONE: No discernible stance and no question asked — pure news sharing, meme, or descriptive post.

## Examples

### Example 1 — Position disclosure with technical reasoning
Post:
Subreddit: r/Forex
Flair: Charts and Setups
Title: GBP/USD double top — I'm short from 1.2740
Body: Price rejected twice at 1.2750. RSI diverging on the H4. SL at 1.2780, TP at 1.2600.

{"reasoning": "POSITION_DISCLOSURE: bearish GBPUSD trade with double-top confirmation.", "content_type": "POSITION_DISCLOSURE", "sarcasm_irony_score": 0, "sentiment_strength": -2, "target_pair": "GBPUSD", "risk_sentiment": "NEUTRAL", "target_clarity": 2, "stance_clarity": "CLEAR"}

### Example 2 — NOISE with question
Post:
Subreddit: r/Forex
Flair: Brokers
Title: Oanda vs IC Markets — which is better for a beginner?

{"reasoning": "NOISE: broker comparison, no market direction.", "content_type": "NOISE", "sarcasm_irony_score": 0, "sentiment_strength": 0, "target_pair": null, "risk_sentiment": "NEUTRAL", "target_clarity": 0, "stance_clarity": "QUESTION"}

### Example 3 — Sarcasm gate
Post:
Subreddit: r/investing
Title: Oh great, Fed raises rates AGAIN. Super bullish for my portfolio 🙄
Body: Can't wait to see what my 401k looks like tomorrow. Love this market.

{"reasoning": "NEWS_REACTION: sarcastic — 'super bullish' is inverted, clear irony.", "content_type": "NEWS_REACTION", "sarcasm_irony_score": 2, "sentiment_strength": 0, "target_pair": "USD", "risk_sentiment": "RISK_OFF", "target_clarity": 1, "stance_clarity": "NONE"}

### Example 4 — Conditional stance
Post:
Subreddit: r/Forex
Flair: Fundamental Analysis
Title: Watching EUR/USD into Thursday's ECB decision
Body: If Lagarde signals another hike and the statement is hawkish I think euro pushes to 1.12 and I'll go long. Otherwise staying flat.

{"reasoning": "FUNDAMENTAL: conditional EUR/USD bullish thesis on ECB outcome.", "content_type": "FUNDAMENTAL", "sarcasm_irony_score": 0, "sentiment_strength": 1, "target_pair": "EURUSD", "risk_sentiment": "NEUTRAL", "target_clarity": 2, "stance_clarity": "CONDITIONAL"}"""

print(f"System prompt: {len(SYSTEM_PROMPT)} chars")

System prompt: 8322 chars


In [ ]:
import re

REQUIRED_FIELDS = {
    "reasoning":           lambda v: isinstance(v, str) and len(v) > 0,
    "content_type":        lambda v: v in ("TECHNICAL", "FUNDAMENTAL", "NEWS_REACTION", "POSITION_DISCLOSURE", "NOISE"),
    "sarcasm_irony_score": lambda v: v in (0, 1, 2),
    "sentiment_strength":  lambda v: isinstance(v, int) and -2 <= v <= 2,
    "target_pair":         lambda v: v is None or isinstance(v, str),
    "risk_sentiment":      lambda v: v in ("RISK_ON", "RISK_OFF", "NEUTRAL"),
    "target_clarity":      lambda v: v in (0, 1, 2),
    "stance_clarity":      lambda v: v in ("CLEAR", "CONDITIONAL", "QUESTION", "NONE"),
}

def validate_labels(labels: dict) -> tuple[bool, str]:
    for field, check in REQUIRED_FIELDS.items():
        if field not in labels:
            return False, f"missing: {field}"
        if not check(labels[field]):
            return False, f"invalid {field}={labels[field]!r}"
    return True, ""

def build_user_message(row: dict) -> str:
    parts = [f"Subreddit: r/{row['subreddit']}"]
    if row.get('flair'):
        parts.append(f"Flair: {row['flair']}")
    parts.append(f"Title: {row['title']}")
    body = str(row.get("body", "")).strip()
    if body:
        parts.append(f"Body: {body[:MAX_BODY_CHARS]}")
    return "\n".join(parts)

_MODEL_BY_PROVIDER = {
    "groq":       lambda: GROQ_MODEL,
    "openrouter": lambda: OPENROUTER_MODEL,
    "google":     lambda: GEMINI_MODEL,
}

def label_post(pool: KeyPool, row: dict) -> tuple[dict, str] | tuple[None, None]:
    user_msg = build_user_message(row)
    attempts = 0

    while attempts < MAX_RETRIES:
        key, provider, client = pool.acquire()
        try:
            raw_text = ""
            if provider == "google":
                genai.configure(api_key=key)
                model = genai.GenerativeModel(
                    model_name=GEMINI_MODEL,
                    system_instruction=SYSTEM_PROMPT
                )
                response = model.generate_content(
                    user_msg,
                    generation_config=genai.GenerationConfig(
                        response_mime_type="application/json",
                        temperature=0.0
                    )
                )
                raw_text = response.text
            elif provider == "openrouter":
                resp = client.chat.completions.create(
                    model=OPENROUTER_MODEL,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": user_msg},
                    ],
                    response_format={"type": "json_object"},
                    temperature=0.0,
                    extra_body={
                        "provider": {
                            "order": ["nextbit"],
                            "allow_fallbacks": False,
                        },
                    }
                )
                raw_text = resp.choices[0].message.content
            else:  # groq
                resp = client.chat.completions.create(
                    model=GROQ_MODEL,
                    messages=[
                        {"role": "system", "content": SYSTEM_PROMPT},
                        {"role": "user",   "content": user_msg},
                    ],
                    response_format={"type": "json_object"},
                    temperature=0.0,
                    max_tokens=1048,
                )
                raw_text = resp.choices[0].message.content

            if not raw_text or not raw_text.strip():
                raise ValueError("Empty response from provider")

            match = re.search(r'\{.*\}', raw_text, re.DOTALL)
            if match:
                raw_text = match.group(0)

            labels = json.loads(raw_text)
            ok, reason = validate_labels(labels)
            if not ok:
                attempts += 1
                tqdm.write(f"  [warn] {reason} - attempt {attempts}")
                continue

            pool.on_success(key)
            model_name = _MODEL_BY_PROVIDER[provider]()
            return labels, model_name

        except RateLimitError as e:
            pool.on_rate_limit(key)
            attempts += 1
        except Exception as e:
            if "429" in str(e):
                pool.on_rate_limit(key)
            else:
                tqdm.write(f"  [error] {provider} ({type(e).__name__}): {e} - attempt {attempts}")
                if 'raw_text' in locals() and raw_text:
                    tqdm.write(f"  [debug] raw_response: {raw_text[:200]}")
                time.sleep(2 ** min(attempts, 4))
            attempts += 1

    return None, None

print("label_post defined — returns (labels, model_name)")

label_post defined — returns (labels, model_name)


In [ ]:
# Test build_user_message on one row per subreddit
for sr, grp in df_clean.groupby("subreddit"):
    row = grp.sample(1, random_state=7).iloc[0].to_dict()
    msg = build_user_message(row)
    print(f"── r/{sr} ──")
    print(msg[:400])
    print()

# Test validate_labels: good and bad cases
good = {
    "reasoning": "Bullish technical analysis on EURUSD with a clear directional call.",
    "content_type": "TECHNICAL",
    "sarcasm_irony_score": 0, "sentiment_strength": 1,
    "target_pair": "EURUSD", "risk_sentiment": "NEUTRAL",
    "target_clarity": 2, "stance_clarity": "CLEAR",
}
ok, reason = validate_labels(good)
print(f"Valid labels → ok={ok}")

# Missing field
bad_missing = {k: v for k, v in good.items() if k != "stance_clarity"}
ok, reason = validate_labels(bad_missing)
print(f"Missing stance_clarity → ok={ok}, reason={reason!r}")

# Empty reasoning string
bad_reasoning = {**good, "reasoning": ""}
ok, reason = validate_labels(bad_reasoning)
print(f"Empty reasoning → ok={ok}, reason={reason!r}")

# Invalid sentiment_strength
bad_strength = {**good, "sentiment_strength": 5}
ok, reason = validate_labels(bad_strength)
print(f"sentiment_strength=5 → ok={ok}, reason={reason!r}")

── r/Forex ──
Subreddit: r/Forex
Flair: Questions
Title: Help
Body: Hi everybody,

Since a year i am into forex trading. I know a lot but there is to much information. I want to take a step back and start again. 

What path do you recommend? 

SMC, ICT, footprints and volume trading. I tried everything but it feels like something is midding.

── r/investing ──
Subreddit: r/investing
Title: Income etfs are sustainable?
Body: I first learned of spyi which has an options strategy for income. I then found out about svol and tltw which have something like a 20% return on their strategies. How can it be so much higher than the risk free rate. If there's a demand for options for which writing them gives a high return. Wouldn't supply expand to give a lesser re

── r/stocks ──
Subreddit: r/stocks
Title: Inflation and 100k+ - what do I do?
Body: Long story short, my wife and I (30, 34) have 119k in the app Acorns. I don’t know much about stocks so this makes it easy (don’t murder me - it’s done

**Message builder and validator work correctly.** Flair is included for Forex and stocks when present, omitted for investing (no flair). Validation catches out-of-range integers and wrong enum values cleanly.

## Main labeling loop

Safe to interrupt and re-run — processed IDs are tracked in the checkpoint file and skipped on resume.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

MAX_WORKERS = 16    # concurrent API calls
BATCH_SIZE  = 80   # futures in flight at once — must be >= MAX_WORKERS

# ── Load checkpoint — backfill model="unknown" for pre-tracking rows ──────────
processed_ids: set[str] = set()
if CHECKPOINT_FILE.exists():
    existing_records = []
    needs_backfill = False
    with open(CHECKPOINT_FILE, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
                if "model" not in rec:
                    rec["model"] = "unknown"
                    needs_backfill = True
                processed_ids.add(rec["id"])
                existing_records.append(rec)
            except (json.JSONDecodeError, KeyError):
                pass
    if needs_backfill:
        with open(CHECKPOINT_FILE, "w", encoding="utf-8") as f:
            for rec in existing_records:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        print(f"Backfilled model='unknown' for {sum(1 for r in existing_records if r['model'] == 'unknown'):,} existing records")
    print(f"Resuming — {len(processed_ids):,} posts already labeled")
else:
    print("Starting fresh")

# ── Build pending work list ───────────────────────────────────────────────────
pending_rows = [
    row.to_dict()
    for _, row in df_labeling.iterrows()
    if str(row["id"]) not in processed_ids
]
print(f"Posts to label: {len(pending_rows):,}")

# ── Worker: runs concurrently in the thread pool ──────────────────────────────
def label_and_record(row_dict: dict) -> dict | None:
    labels, model_name = label_post(pool, row_dict)
    if labels is None:
        return None
    return {
        "id":          str(row_dict["id"]),
        "subreddit":   row_dict["subreddit"],
        "title":       row_dict["title"],
        "body":        row_dict["body"],
        "flair":       row_dict["flair"],
        "score":       row_dict["score"],
        "created_utc": row_dict["created_utc"],
        "model":       model_name,
        **labels,
    }

# ── Batched parallel loop — Colab-friendly ────────────────────────────────────
# Submitting all 166k futures at once causes a multi-minute setup delay.
# Batching keeps only BATCH_SIZE futures in flight — progress shows immediately.
pool    = KeyPool(API_KEYS)
labeled = 0
skipped = 0

with open(CHECKPOINT_FILE, "a", encoding="utf-8") as out_f:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        pbar = tqdm(total=len(df_labeling), desc="Labeling")
        pbar.update(len(processed_ids))

        for batch_start in range(0, len(pending_rows), BATCH_SIZE):
            batch   = pending_rows[batch_start : batch_start + BATCH_SIZE]
            futures = [executor.submit(label_and_record, row) for row in batch]

            for future in as_completed(futures):
                record = future.result()
                if record is None:
                    skipped += 1
                else:
                    out_f.write(json.dumps(record, ensure_ascii=False) + "\n")
                    labeled += 1
                    if labeled % CHECKPOINT_EVERY == 0:
                        out_f.flush()
                        tqdm.write(f"  [checkpoint] {labeled:,} labeled | {skipped} skipped")
                pbar.update(1)
                pbar.set_postfix(labeled=labeled, skipped=skipped)

        pbar.close()

print(f"\nDone — {labeled:,} labeled | {skipped} skipped")

Resuming — 24,338 posts already labeled
Posts to label: 144,399


Labeling:   0%|          | 0/168737 [00:00<?, ?it/s]

  [checkpoint] 40 labeled | 0 skipped
  [error] openrouter (TypeError): 'NoneType' object is not subscriptable - attempt 0
  [error] openrouter (TypeError): 'NoneType' object is not subscriptable - attempt 0
  [error] openrouter (TypeError): 'NoneType' object is not subscriptable - attempt 0
  [checkpoint] 80 labeled | 0 skipped
  [checkpoint] 120 labeled | 0 skipped
  [error] openrouter (TypeError): 'NoneType' object is not subscriptable - attempt 0
  [error] openrouter (TypeError): 'NoneType' object is not subscriptable - attempt 0
  [checkpoint] 160 labeled | 0 skipped
  [checkpoint] 200 labeled | 0 skipped
  [checkpoint] 240 labeled | 0 skipped
  [checkpoint] 280 labeled | 0 skipped
  [checkpoint] 320 labeled | 0 skipped
  [checkpoint] 360 labeled | 0 skipped
  [error] openrouter (ValueError): Empty response from provider - attempt 0
  [checkpoint] 400 labeled | 0 skipped
  [checkpoint] 440 labeled | 0 skipped
  [checkpoint] 480 labeled | 0 skipped
  [checkpoint] 520 labeled | 0 sk

KeyboardInterrupt: 

## Save & validate output

In [ ]:
timestamp   = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = OUTPUT_DIR / f"reddit_labeled_{timestamp}.jsonl"
shutil.copy(CHECKPOINT_FILE, output_file)
print(f"Saved: {output_file}")

df_out = pd.read_json(output_file, lines=True)
print(f"Shape: {df_out.shape}")

ordinal_cols     = ["sentiment_strength", "target_clarity", "sarcasm_irony_score"]
categorical_cols = ["content_type", "stance_clarity", "risk_sentiment"]

for col in ordinal_cols:
    print(f"\n--- {col} ---")
    print(df_out[col].value_counts().sort_index())

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print(df_out[col].value_counts())

n_sarcastic = (df_out["sarcasm_irony_score"] == 2).sum()
print(f"\nPosts with sarcasm_irony_score=2 (labels masked in training): {n_sarcastic:,} ({n_sarcastic/len(df_out):.1%})")
print(f"target_pair null: {df_out['target_pair'].isna().sum():,} ({df_out['target_pair'].isna().mean():.1%})")